<a href="https://colab.research.google.com/github/lilara4/From-Shelves-to-Analytics/blob/main/Clean_SuperStore_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("Sample Superstore.csv", encoding='latin-1')
print(df.head(20))

FileNotFoundError: [Errno 2] No such file or directory: 'Sample Superstore.csv'

In [ ]:
print(">> Summary Statistics ")
print(df[['Sales', 'Quantity', 'Discount', 'Profit']].describe())
print("\n")

print(">> Correlation Matrix ")
corr = df[['Sales', 'Quantity', 'Discount', 'Profit']].corr()
print(corr)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='copper')
plt.title('Correlation Heatmap')
plt.show()

print(">> Key Metrics")
print("\nMissing Values per Column:")
print(df.isnull().sum())

categorical_cols = ['Order ID', 'Ship Mode', 'Customer ID', 'Customer Name',
                    'Segment', 'Country', 'City', 'State', 'Region',
                    'Product ID', 'Category', 'Sub-Category', 'Product Name']
for col in categorical_cols:
    print(f"\nValue Counts for {col}:")
    print(df[col].value_counts())

numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
df[numeric_cols].hist(figsize=(12,8), bins=20, color='#D2B48C')
plt.tight_layout()
plt.show()

In [ ]:
# Summary of issues
summary = {
    "missing_values": df.isnull().sum(),
    "duplicates": df.duplicated().sum(),
    "data_types": df.dtypes,
    "discount_out_of_range": df[(df['Discount'] < 0) | (df['Discount'] > 1)].shape[0],
    "date_check": (df['Ship Date'] < df['Order Date']).sum()
}

summary

In [ ]:
# Convert Order Date and Ship Date to datetime format
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')

# Check missing values before cleaning
print("Missing values BEFORE cleaning:")
print(df[['Order Date', 'Ship Date']].isna().sum())
print("\n")

#Fix Ship Date Using Median Shippig Days
valid_dates = df[(df['Ship Date'].notna())&
                 (df['Order Date'].notna())&
                 (df['Ship Date'] >= df['Order Date'] )]

shipping_days = (valid_dates['Ship Date']- valid_dates['Order Date']).dt.days

median_days = shipping_days.median()
print("media shipping days:", median_days)

df.loc[df['Ship Date'] < df['Order Date'] , 'Ship Date'] =df['Order Date']+ pd.Timedelta(days=median_days)
df.loc[df['Ship Date'].isna(),'Ship Date']=df['Order Date']+pd.Timedelta(days=median_days)


# Check missing values after cleaning
print("Missing values AFTER cleaning:")
print(df[['Order Date', 'Ship Date']].isna().sum())
print("\n")

# Data quality validation report
print("Data Quality Report:")
print("-"*40)

# Count logical date errors
errors = (df['Ship Date'] < df['Order Date']).sum()
print(f"Ship Date earlier than Order Date: {errors}")

# Check data types of date columns
print("Date types check:")
print(df[['Order Date', 'Ship Date']].dtypes)

# Display sample of cleaned data
print("\nSample cleaned rows:")
print(df[['Order Date', 'Ship Date']].head(20))

# Save cleaned dataset to CSV file
df.to_csv('results.csv', index=False)


In [ ]:
# Select numeric columns for outlier detection
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

# Dictionary to store outlier summary for each column
outliers_summary = {}

# Loop through each numeric column
for col in numeric_cols:

    # Calculate Q1, Q3, and IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    # Define lower and upper bounds using IQR method
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    # Store outlier information
    outliers_summary[col] = {
        'count': outliers.shape[0],
        'lower_bound': lower_bound,
        'upper_bound': upper_bound
    }

    # Plot boxplot to visualize outliers
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[col], color ='#E8D5B7')
    plt.title(f'Boxplot of {col}')
    plt.show()

# Print summary of detected outliers
for col, info in outliers_summary.items():
    print(f"{col}: {info['count']} outliers, "
          f"Lower bound = {info['lower_bound']}, "
          f"Upper bound = {info['upper_bound']}")

In [ ]:
# Numeric columns to handle outliers
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

# Dictionary to store lower and upper bounds
bounds = {}

# Calculate IQR bounds for each numeric column
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    bounds[col] = (lower_bound, upper_bound)

# Handle outliers by clipping values to acceptable ranges


# Quantity should be at least 1
df['Quantity'] = df['Quantity'].clip(lower=1, upper=int(bounds['Quantity'][1]))

# Discount is limited to a reasonable range (0 to 50%)
df['Discount'] = df['Discount'].clip(lower=0, upper=0.5)


# Visualize data after outlier handling
for col in numeric_cols:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[col],color ='#E8D5B7')
    plt.title(f'Boxplot of {col} after outlier handling')
    plt.show()

# Print min and max values after cleaning
for col in numeric_cols:
    print(f"{col}: min = {df[col].min()}, max = {df[col].max()}")

# Save the cleaned dataset
df.to_csv('Samplemy_Superstore_clean_simple.csv', index=False)
